# Homework 2 Solution

1. Save a copy of this `.ipynb` file to your Google Drive or PC.  
2. Rename the file from **`HW2.ipynb`** to **`HW2_(your name).ipynb`**.  
3. Complete the code cells below by following the given instructions.  
4. Save your work and upload the `.ipynb` file to Canvas.  
   - Do **NOT** clear the outputs; leave your results visible.  
5. ⚠️ **IMPORTANT**: Convert your `.ipynb` file to a `.PDF` and upload them **together**.
    - Go to the top menu → **File → Print → Save as PDF**.  


.

.


# [HW 2-1] FFT-based Feature Extraction
1. Load `ExampleData` (Data link: 'https://github.com/ljwg3000/UNT_MEEN-AI-Fall2026/blob/main/AI_tutorial/DA1/ExampleData?raw=true').  
2. Compute the **single-sided FFT spectrum** for each signal (acceleration, voltage, current).  
3. On the positive frequencies, find the **top-2 peaks** (frequency in Hz and magnitude).  
4. Build a `4×3 NumPy array` in this row order:  
   - Row 1: largest-peak frequency (Hz)  
   - Row 2: largest-peak magnitude  
   - Row 3: second-peak frequency (Hz)  
   - Row 4: second-peak magnitude  
5. **Transpose** to **3×4**, convert to a **DataFrame**, name the columns  
   `['peak1_freq_Hz','peak1_amp','peak2_freq_Hz','peak2_amp']`,  
   and index the rows as `['acceleration','voltage','current']`.  
   - Using `df.index = [' ',' ', ...]` to apply indices of row.
6. Print the resulting DataFrame.

.

👉  Refer to `DA1_Code3`, `DA2_Code1`

### Notes for HW 2-1

- A **peak** here means a local maximum returned by `find_peaks`. Rank these peaks by their single-sided amplitudes; do not select DC (0 Hz).
- The lesson uses `height=0.5`. That threshold would exclude the acceleration signal's largest peaks in this dataset, so this solution changes only the threshold to `0`.
- After saving the first peak, set its amplitude to zero in a temporary copy. Applying `np.argmax` again selects the second peak. All three signals have at least two positive local peaks.
- **One correction to the lesson FFT code:** this dataset has an even number of samples, and `freq[:N//2]` excludes the Nyquist bin. Its last retained bin is therefore an ordinary positive-frequency bin. The line `amp[-1] = amp[-1] / 2` is omitted; only DC is halved. This correction does not change the two largest peaks for this dataset.
- Peak frequencies are reported at the FFT bins, so values near 60, 120, 180, or 240 Hz will not be exact integers.

In [ ]:
# Import the necessary packages
# From DA2_Code1: FFT and find_peaks examples.
import pandas as pd
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks

# Load ExampleData from Github
Data = pd.read_csv('https://github.com/ljwg3000/UNT_MEEN-AI-Fall2026/blob/main/AI_tutorial/DA1/ExampleData?raw=true', sep=',', header=None)

# Select each sensor data
t = Data.iloc[:, 0].values
acc = Data.iloc[:, 1].values
volt = Data.iloc[:, 2].values
curr = Data.iloc[:, 3].values

# Implement FFT for each sensor data
# From DA2_Code1: calculate the sampling frequency from the time column.
dt = np.mean(np.diff(t))
fs = 1.0 / dt

# Repeat the same FFT code three times and save each result.
# The slice [:N//2] excludes Nyquist for this even-length dataset.
# Therefore, do not include the lesson's amp[-1] = amp[-1] / 2 line.

# Acceleration
x = acc
N = len(x)
X = fft(x)
freq = fftfreq(N, 1/fs)

# single-sided
k_pos = N//2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling
amp = (2 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2

f_pos_acc = f_pos
amp_acc = amp

# Voltage
x = volt
N = len(x)
X = fft(x)
freq = fftfreq(N, 1/fs)

# single-sided
k_pos = N//2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling
amp = (2 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2

f_pos_volt = f_pos
amp_volt = amp

# Current
x = curr
N = len(x)
X = fft(x)
freq = fftfreq(N, 1/fs)

# single-sided
k_pos = N//2
f_pos = freq[:k_pos]
X_pos = X[:k_pos]

# Amplitude scaling
amp = (2 / N) * np.abs(X_pos)
amp[0] = amp[0] / 2

f_pos_curr = f_pos
amp_curr = amp

# Extract Peak features
# From DA2_Code1: find_peaks identifies local maxima.
# Change height=0.5 to height=0 so small acceleration peaks are included.
# DC is at an endpoint and is not returned by find_peaks.
# From DA2_Code1: repeat np.argmax to select the two largest peaks.
# From DA2_Code3: use .copy() to keep the original values unchanged.

# Acceleration
peaks, props = find_peaks(amp_acc, height=0)
peak_freqs = f_pos_acc[peaks]
peak_amps = amp_acc[peaks]

# Largest peak
peak_idx = np.argmax(peak_amps)
peak1_freq_acc = peak_freqs[peak_idx]
peak1_amp_acc = peak_amps[peak_idx]

# Second-largest peak: set the first peak to zero in a temporary copy,
# then repeat the same maximum-peak code.
peak_amps_temp = peak_amps.copy()
peak_amps_temp[peak_idx] = 0
peak_idx = np.argmax(peak_amps_temp)
peak2_freq_acc = peak_freqs[peak_idx]
peak2_amp_acc = peak_amps[peak_idx]

# Voltage
peaks, props = find_peaks(amp_volt, height=0)
peak_freqs = f_pos_volt[peaks]
peak_amps = amp_volt[peaks]

# Largest peak
peak_idx = np.argmax(peak_amps)
peak1_freq_volt = peak_freqs[peak_idx]
peak1_amp_volt = peak_amps[peak_idx]

# Second-largest peak: set the first peak to zero in a temporary copy,
# then repeat the same maximum-peak code.
peak_amps_temp = peak_amps.copy()
peak_amps_temp[peak_idx] = 0
peak_idx = np.argmax(peak_amps_temp)
peak2_freq_volt = peak_freqs[peak_idx]
peak2_amp_volt = peak_amps[peak_idx]

# Current
peaks, props = find_peaks(amp_curr, height=0)
peak_freqs = f_pos_curr[peaks]
peak_amps = amp_curr[peaks]

# Largest peak
peak_idx = np.argmax(peak_amps)
peak1_freq_curr = peak_freqs[peak_idx]
peak1_amp_curr = peak_amps[peak_idx]

# Second-largest peak: set the first peak to zero in a temporary copy,
# then repeat the same maximum-peak code.
peak_amps_temp = peak_amps.copy()
peak_amps_temp[peak_idx] = 0
peak_idx = np.argmax(peak_amps_temp)
peak2_freq_curr = peak_freqs[peak_idx]
peak2_amp_curr = peak_amps[peak_idx]

# From DA1_Code3: create an empty array and fill its entries.
# Rows: peak1 frequency, peak1 amplitude, peak2 frequency, peak2 amplitude.
# Columns: acceleration, voltage, current.
Feature_arr = np.zeros((4, 3))

Feature_arr[0, 0] = peak1_freq_acc
Feature_arr[1, 0] = peak1_amp_acc
Feature_arr[2, 0] = peak2_freq_acc
Feature_arr[3, 0] = peak2_amp_acc

Feature_arr[0, 1] = peak1_freq_volt
Feature_arr[1, 1] = peak1_amp_volt
Feature_arr[2, 1] = peak2_freq_volt
Feature_arr[3, 1] = peak2_amp_volt

Feature_arr[0, 2] = peak1_freq_curr
Feature_arr[1, 2] = peak1_amp_curr
Feature_arr[2, 2] = peak2_freq_curr
Feature_arr[3, 2] = peak2_amp_curr

# Transform into DataFrame and print it
# HW2, Step 5: .T transposes the array from 4 x 3 to 3 x 4.
Feature_arr_T = Feature_arr.T
Feature_df = pd.DataFrame(Feature_arr_T)

# HW2, Step 5: assign the required column and row names.
Feature_df.columns = ['peak1_freq_Hz', 'peak1_amp', 'peak2_freq_Hz', 'peak2_amp']
Feature_df.index = ['acceleration', 'voltage', 'current']
Feature_df


,peak1_freq_Hz,peak1_amp,peak2_freq_Hz,peak2_amp
acceleration,239.943014,0.387202,119.971507,0.359885
voltage,59.985753,1.465263,179.957260,0.924527
current,59.985753,3.448837,179.957260,1.485710


.

.

# [HW 2-2] Wavelet-based features (acceleration, level=7)

1. From `ExampleData`, take the **acceleration** signal.  
2. Perform **7-level** discrete wavelet decomposition with `pywt.wavedec` (e.g., `'db4'`).  
3. Convert the coefficient list into a convenient table (DataFrame).    
4. For **each coefficient** compute **six features**:  
   - max, min, RMS, variance, standard deviation, mean.  
5. Assemble a **6×8 array** (rows = the six features; cols = [a7, d7, …, d1]).  
6. Convert it to a DataFrame and print.

.

👉  Refer to `DA1_Code3`, `DA2_Code3`

### Notes for HW 2-2

- `pywt.wavedec` returns coefficients in the order `[a7, d7, d6, d5, d4, d3, d2, d1]`.
- The coefficient arrays have different lengths. Converting the list to a DataFrame adds `NaN` padding to shorter rows. Keep that table for inspection and calculate features directly from `Coefficient[i]`.
- The feature order remains the same as DA1_Code3: max, min, RMS, variance, standard deviation, and mean. NumPy's default variance and standard deviation definitions are retained.
- Only the final table labels are added to make the 6 x 8 result easier to read.

In [ ]:
# Import (additional) packages
# From DA2_Code3.
import pywt

# WT-based signal decomposition
# From DA2_Code3: change Level from 8 to 7 and select acceleration (column 1).
MotherWavelet = pywt.Wavelet('db4')   # Mother wavelet
Level = 7                          # Wavelet levels

Data_Target = Data.iloc[:,1] # Select the acceleration signal
Coefficient = pywt.wavedec(Data_Target, MotherWavelet, level=Level, axis=0)

# Confirm extracted coefficients as DataFrame (DA2_Code3).
# The order is [a7, d7, d6, d5, d4, d3, d2, d1].
Coefficient_df = pd.DataFrame(Coefficient)
Coefficient_df.index = ['a7', 'd7', 'd6', 'd5', 'd4', 'd3', 'd2', 'd1']
print(Coefficient_df)

# Feature extraction
# Copy the RMS function directly from DA1_Code3.
def rms(a):
    return np.sqrt(np.mean(a**2))

# From DA1_Code3: change the array size from (6, number_of_sensors) to (6, 8).
WT_Feature_arr = np.zeros((6, 8))

# Copy the six-feature loop from DA1_Code3.
# Change the input from ExampleData.iloc[:,i+1] to Coefficient[i].
# Use the original coefficient arrays so the table's NaN padding is excluded.
for i in range(8):
    WT_Feature_arr[0,i] = np.max(Coefficient[i])
    WT_Feature_arr[1,i] = np.min(Coefficient[i])
    WT_Feature_arr[2,i] =    rms(Coefficient[i])
    WT_Feature_arr[3,i] = np.var(Coefficient[i])
    WT_Feature_arr[4,i] = np.std(Coefficient[i])
    WT_Feature_arr[5,i] = np.mean(Coefficient[i])

# Convert to a DataFrame, keeping the required 6 x 8 orientation.
WT_Feature_df = pd.DataFrame(WT_Feature_arr)
WT_Feature_df.columns = ['a7', 'd7', 'd6', 'd5', 'd4', 'd3', 'd2', 'd1']
WT_Feature_df.index = ['Max', 'Min', 'RMS', 'Variance', 'Std', 'Mean']
WT_Feature_df


        0         1         2     ...      1387      1388      1389
a7  0.202077  0.141074  0.241260  ...       NaN       NaN       NaN
d7 -0.008568 -0.038576  0.005313  ...       NaN       NaN       NaN
d6 -0.004133 -0.006233 -0.267588  ...       NaN       NaN       NaN
d5  0.001655 -0.000542  0.073128  ...       NaN       NaN       NaN
d4  0.000409  0.000799  0.000564  ...       NaN       NaN       NaN
d3  0.000206  0.000374 -0.005132  ...       NaN       NaN       NaN
d2  0.000292  0.000339 -0.006453  ...       NaN       NaN       NaN
d1  0.000540  0.006674  0.008901  ... -0.009993 -0.020788  0.031757

[8 rows x 1390 columns]


,a7,d7,d6,d5,d4,d3,d2,d1
Max,0.717820,2.565467,3.944576,3.507259,1.305028,0.993955,0.290266,0.185770
Min,-0.126120,-1.959290,-4.568351,-3.201902,-1.410950,-0.961073,-0.439236,-0.262644
RMS,0.239670,1.134179,2.010050,1.493542,0.437699,0.253681,0.065724,0.047869
Variance,0.032840,1.286118,4.039305,2.230256,0.190106,0.064353,0.004319,0.002291
Std,0.181217,1.134071,2.009802,1.493404,0.436011,0.253679,0.065722,0.047869
Mean,0.156851,0.015630,-0.031590,0.020317,0.038407,-0.001010,-0.000483,-0.000012
